# LeetCode 2160 — Minimum Sum of Four Digit Number After Splitting Digits bằng Python

**Link đề:** https://leetcode.com/problems/minimum-sum-of-four-digit-number-after-splitting-digits/
**Độ khó:** Easy · **Chủ đề:** Math, Greedy, Sorting

```python
class Solution:
    def minimumSum(self, num: int) -> int:
        ...
```

## Mục tiêu học
1. Tách chữ số của một số bằng hai cách: **chuỗi** (`str`, `int`) và **toán** (`% 10`, `// 10`).
2. Brute force có hệ thống: thử mọi **hoán vị** chữ số và mọi điểm cắt.
3. Rút ra insight **giá trị vị trí (place value)**: chữ số ở hàng chục "nặng" gấp 10 lần hàng đơn vị → đặt chữ số nhỏ vào hàng chục.
4. Lời giải greedy: sắp chữ số tăng dần, `(d0·10 + d2) + (d1·10 + d3)`, và công thức gọn `10·(d0 + d1) + d2 + d3`.
5. Chứng minh vì sao tách thành **hai số có hai chữ số** là tối ưu.

## 1. Đề bài

Cho số nguyên dương `num` có **đúng 4 chữ số**. Dùng **tất cả** 4 chữ số đó để tạo hai số mới `new1`, `new2` (được phép có **số 0 ở đầu**). Trả về **tổng nhỏ nhất** có thể của `new1 + new2`.

Ví dụ với `num = 2932` (chữ số: 2, 9, 3, 2), một vài cặp hợp lệ: `[22, 93]`, `[23, 92]`, `[223, 9]`, `[2, 329]`.

| `num` | Kết quả | Cặp tối ưu |
|:-:|:-:|---|
| `2932` | `52` | `29 + 23` |
| `4009` | `13` | `04 + 09` = `4 + 9` |

**Constraints:** `1000 <= num <= 9999`.

"Số 0 ở đầu được phép" nghĩa là `"04"` được hiểu là `4` — rất quan trọng với `4009`.

In [1]:
num = 2932
for pair in ([22, 93], [23, 92], [223, 9], [2, 329], [29, 23]):
    print(pair, "-> tổng", sum(pair))

[22, 93] -> tổng 115
[23, 92] -> tổng 115
[223, 9] -> tổng 232
[2, 329] -> tổng 331
[29, 23] -> tổng 52


## 2. Nền tảng Python cần biết

### 2.1. Tách chữ số bằng chuỗi

- `str(num)` → `"2932"`; duyệt chuỗi được từng ký tự `'2'`, `'9'`, ...
- `int('9')` → `9` (đổi ký tự số thành số).
- `[int(c) for c in str(num)]` → list chữ số.

In [2]:
num = 2932
s = str(num)
print(s, type(s))
digits = [int(c) for c in s]
print(digits)
print(int("04"), "<- int bỏ qua số 0 ở đầu")

2932 <class 'str'>
[2, 9, 3, 2]
4 <- int bỏ qua số 0 ở đầu


### 2.2. Tách chữ số bằng toán: `% 10` và `// 10`

- `num % 10` → chữ số **cuối**.
- `num // 10` → bỏ chữ số cuối (chia lấy phần nguyên).
- Lặp 4 lần sẽ lấy đủ 4 chữ số (theo thứ tự **ngược**).

In [3]:
num = 2932
digits = []
x = num
while x > 0:
    digits.append(x % 10)
    print(f"x={x:>4} -> chữ số cuối {x % 10}, còn lại {x // 10}")
    x //= 10
print("digits (ngược):", digits)

x=2932 -> chữ số cuối 2, còn lại 293
x= 293 -> chữ số cuối 3, còn lại 29
x=  29 -> chữ số cuối 9, còn lại 2
x=   2 -> chữ số cuối 2, còn lại 0
digits (ngược): [2, 3, 9, 2]


Lưu ý: với `4009`, cách toán học vẫn lấy đủ các chữ số 0 ở giữa vì vòng chạy tới khi `x == 0`, mà `4009 → 400 → 40 → 4 → 0` đi qua đủ 4 bước.

### 2.3. `sorted` trên list chữ số và unpacking

`a, b, c, d = sorted(digits)` gán 4 phần tử của list đã sắp cho 4 biến (unpacking). Số biến phải **bằng** số phần tử.

In [4]:
d0, d1, d2, d3 = sorted([2, 9, 3, 2])
print(d0, d1, d2, d3)
try:
    a, b = [1, 2, 3]
except ValueError as e:
    print("ValueError:", e)

2 2 3 9
ValueError: too many values to unpack (expected 2)


### 2.4. Giá trị vị trí (place value)

Số hai chữ số `"ab"` có giá trị `10·a + b`. Chữ số hàng chục **nhân 10**, hàng đơn vị nhân 1. Số ba chữ số `"abc"` = `100·a + 10·b + c`.

In [5]:
a, b_ = 2, 9
print("ab =", 10 * a + b_, "| ba =", 10 * b_ + a)
print("int('293') =", int("293"), "= 100*2 + 10*9 + 3 =", 100 * 2 + 10 * 9 + 3)

ab = 29 | ba = 92
int('293') = 293 = 100*2 + 10*9 + 3 = 293


### 2.5. `itertools.permutations`

`permutations(dãy)` sinh mọi cách **sắp thứ tự** các phần tử (4 chữ số → `4! = 24` hoán vị, có thể trùng nếu chữ số trùng).

In [6]:
from itertools import permutations
perms = list(permutations("123"))
print(len(perms), perms[:4])
print(["".join(p) for p in perms])

6 [('1', '2', '3'), ('1', '3', '2'), ('2', '1', '3'), ('2', '3', '1')]
['123', '132', '213', '231', '312', '321']


## 3. Cách 1 — Brute force: mọi hoán vị × mọi điểm cắt

Mọi cách chia 4 chữ số thành hai số đều có thể mô tả bằng: (1) một thứ tự của 4 chữ số; (2) một điểm cắt `p` ∈ {1, 2, 3}: `new1` = `p` chữ số đầu, `new2` = phần còn lại.

In [7]:
from itertools import permutations

def minimum_sum_bruteforce(num: int) -> int:
    s = str(num)
    best = None
    for perm in permutations(s):
        t = "".join(perm)
        for p in range(1, len(t)):          # điểm cắt 1, 2, 3
            total = int(t[:p]) + int(t[p:])
            if best is None or total < best:
                best = total
    return best

print(minimum_sum_bruteforce(2932))   # 52
print(minimum_sum_bruteforce(4009))   # 13

52
13


### Chi phí brute force

`4! · 3 = 72` cách — với 4 chữ số thì **brute force hoàn toàn chấp nhận được** (hằng số). Nhưng nó không cho ta *hiểu* vì sao đáp án là 52, và nếu đề tăng lên `d` chữ số thì có `d! · (d - 1)` cách — bùng nổ nhanh.

Liệt kê thử các ứng viên tốt nhất để tìm quy luật:

In [8]:
cands = set()
for perm in permutations("2932"):
    t = "".join(perm)
    for p in range(1, 4):
        cands.add((int(t[:p]) + int(t[p:]), t[:p], t[p:]))
for total, a, b_ in sorted(cands)[:6]:
    print(total, a, "+", b_)
print("...")
for total, a, b_ in sorted(cands)[-3:]:
    print(total, a, "+", b_)

52 23 + 29
52 29 + 23
61 22 + 39
61 29 + 32
61 32 + 29
61 39 + 22
...
925 923 + 2
934 2 + 932
934 932 + 2


## 4. Insight: hai số hai chữ số, chữ số nhỏ lên hàng chục

**Quan sát 1 — đừng tạo số ba chữ số.** Nếu một số có 3 chữ số `"abc"` và số kia 1 chữ số `"d"`, tổng `100a + 10b + c + d`. Chuyển `a` sang làm hàng chục của số kia: `"bc" + "ad"` = `10b + c + 10a + d` — hàng trăm của `a` (×100) giảm xuống ×10, nên tổng **không tăng** (giảm `90a`).

**Quan sát 2 — hàng chục "nặng" gấp 10.** Với hai số `"ab"`, `"cd"`: tổng `= 10(a + c) + (b + d)`. Muốn nhỏ → hai chữ số ở hàng chục (`a`, `c`) phải là **hai chữ số nhỏ nhất**, hai chữ số lớn nhất xuống hàng đơn vị.

→ Sắp `d0 ≤ d1 ≤ d2 ≤ d3`: đáp án `= 10·(d0 + d1) + (d2 + d3)`.

Còn ghép cụ thể là `d0d2 + d1d3` hay `d0d3 + d1d2` đều **cùng tổng**.

In [9]:
d0, d1, d2, d3 = sorted([2, 9, 3, 2])
print("sorted:", d0, d1, d2, d3)
print("d0d2 + d1d3 =", 10 * d0 + d2, "+", 10 * d1 + d3, "=", 10 * d0 + d2 + 10 * d1 + d3)
print("d0d3 + d1d2 =", 10 * d0 + d3, "+", 10 * d1 + d2, "=", 10 * d0 + d3 + 10 * d1 + d2)
print("công thức 10*(d0+d1)+(d2+d3) =", 10 * (d0 + d1) + d2 + d3)

sorted: 2 2 3 9
d0d2 + d1d3 = 23 + 29 = 52
d0d3 + d1d2 = 29 + 23 = 52
công thức 10*(d0+d1)+(d2+d3) = 52


### Thí nghiệm: chọn hai chữ số nào cho hàng chục?

Với dạng (2+2), tổng chỉ phụ thuộc **tập** hai chữ số ở hàng chục. Có `C(4,2) = 6` cách chọn hai vị trí (trong list đã sắp) làm hàng chục. Ta liệt kê để thấy công thức `9·(t1 + t2) + S` (với `S` = tổng 4 chữ số):

In [10]:
from itertools import combinations
d = sorted([2, 9, 3, 2])
S = sum(d)
print("d =", d, "| S =", S)
for tens_idx in combinations(range(4), 2):
    tens = [d[i] for i in tens_idx]
    units = [d[i] for i in range(4) if i not in tens_idx]
    total = 10 * sum(tens) + sum(units)
    print(f"hàng chục {tens}, hàng đơn vị {units}: tổng = {total} = 9*{sum(tens)} + {S}")

d = [2, 2, 3, 9] | S = 16
hàng chục [2, 2], hàng đơn vị [3, 9]: tổng = 52 = 9*4 + 16
hàng chục [2, 3], hàng đơn vị [2, 9]: tổng = 61 = 9*5 + 16
hàng chục [2, 9], hàng đơn vị [2, 3]: tổng = 115 = 9*11 + 16
hàng chục [2, 3], hàng đơn vị [2, 9]: tổng = 61 = 9*5 + 16
hàng chục [2, 9], hàng đơn vị [2, 3]: tổng = 115 = 9*11 + 16
hàng chục [3, 9], hàng đơn vị [2, 2]: tổng = 124 = 9*12 + 16


Tổng nhỏ nhất khi hai chữ số hàng chục là **hai chữ số nhỏ nhất** (`[2, 2]` → 52). `combinations(dãy, 2)` sinh mọi cách chọn 2 phần tử **không quan tâm thứ tự** (khác `permutations`).

## 5. Cách 2 — Lời giải chính: sắp chữ số rồi ghép

Từng đoạn:

**Đoạn 1:** lấy 4 chữ số: `digits = [int(c) for c in str(num)]`.

**Đoạn 2:** sắp tăng dần: `d = sorted(digits)`.

**Đoạn 3:** ghép `new1 = d[0]*10 + d[2]`, `new2 = d[1]*10 + d[3]`.

**Đoạn 4:** trả `new1 + new2`.

In [11]:
def minimum_sum_sort(num: int) -> int:
    digits = [int(c) for c in str(num)]   # Đoạn 1
    d = sorted(digits)                     # Đoạn 2
    new1 = d[0] * 10 + d[2]                # Đoạn 3
    new2 = d[1] * 10 + d[3]
    return new1 + new2                     # Đoạn 4

print(minimum_sum_sort(2932), minimum_sum_sort(4009), minimum_sum_sort(1000), minimum_sum_sort(9999))

52 13 1 198


## 6. TRACE từng bước

| `num` | chữ số | sau sort | `new1 = d0d2` | `new2 = d1d3` | tổng |
|:-:|---|---|:-:|:-:|:-:|
| 2932 | 2,9,3,2 | 2,2,3,9 | 23 | 29 | **52** |
| 4009 | 4,0,0,9 | 0,0,4,9 | 04 = 4 | 09 = 9 | **13** |
| 1000 | 1,0,0,0 | 0,0,0,1 | 00 = 0 | 01 = 1 | **1** |

In [12]:
def trace_minimum_sum(num):
    digits = [int(c) for c in str(num)]
    d = sorted(digits)
    new1 = d[0] * 10 + d[2]
    new2 = d[1] * 10 + d[3]
    print(f"num={num}: digits={digits} -> sorted={d}")
    print(f"   new1 = {d[0]}*10 + {d[2]} = {new1:>2} (viết '{d[0]}{d[2]}')")
    print(f"   new2 = {d[1]}*10 + {d[3]} = {new2:>2} (viết '{d[1]}{d[3]}')")
    print(f"   tổng = {new1 + new2}")
    return new1 + new2

for num in (2932, 4009, 1000):
    trace_minimum_sum(num)

num=2932: digits=[2, 9, 3, 2] -> sorted=[2, 2, 3, 9]
   new1 = 2*10 + 3 = 23 (viết '23')
   new2 = 2*10 + 9 = 29 (viết '29')
   tổng = 52
num=4009: digits=[4, 0, 0, 9] -> sorted=[0, 0, 4, 9]
   new1 = 0*10 + 4 =  4 (viết '04')
   new2 = 0*10 + 9 =  9 (viết '09')
   tổng = 13
num=1000: digits=[1, 0, 0, 0] -> sorted=[0, 0, 0, 1]
   new1 = 0*10 + 0 =  0 (viết '00')
   new2 = 0*10 + 1 =  1 (viết '01')
   tổng = 1


## 7. Code nộp LeetCode — Cách 2

In [13]:
class Solution:
    def minimumSum(self, num: int) -> int:
        d = sorted(int(c) for c in str(num))
        return (d[0] * 10 + d[2]) + (d[1] * 10 + d[3])

sol = Solution()
print(sol.minimumSum(2932), sol.minimumSum(4009))

52 13


`sorted(...)` nhận được cả **generator** `int(c) for c in str(num)` và luôn trả về **list**.

## 8. Cách 3 — Công thức + tách chữ số bằng toán (không dùng chuỗi)

Dùng `% 10` và `// 10` để lấy chữ số, rồi công thức `10·(d0 + d1) + d2 + d3`. Dùng `divmod(x, 10)` trả cùng lúc `(x // 10, x % 10)`.

In [14]:
print(divmod(2932, 10))

(293, 2)


In [15]:
def minimum_sum_math(num: int) -> int:
    digits = []
    for _ in range(4):
        num, r = divmod(num, 10)
        digits.append(r)
    d0, d1, d2, d3 = sorted(digits)
    return 10 * (d0 + d1) + d2 + d3

print(minimum_sum_math(2932), minimum_sum_math(4009))

52 13


In [16]:
class Solution:
    def minimumSum(self, num: int) -> int:
        digits = []
        for _ in range(4):
            num, r = divmod(num, 10)
            digits.append(r)
        d0, d1, d2, d3 = sorted(digits)
        return 10 * (d0 + d1) + d2 + d3

print(Solution().minimumSum(2932), Solution().minimumSum(4009))

52 13


Dùng `for _ in range(4)` (đúng 4 lần) thay vì `while num > 0` để code thể hiện rõ "số có đúng 4 chữ số". Dấu `_` là tên biến quy ước cho "không dùng tới".

## 9. Cách 4 — Đếm chữ số (counting) cho trường hợp tổng quát

Ý tưởng mở rộng cho số có `d` chữ số (biến thể): đếm số lần xuất hiện của mỗi chữ số `0..9`, rồi **phân phát** lần lượt từ chữ số nhỏ nhất, xen kẽ cho hai số: số nhỏ nhất vào hàng cao nhất của số 1, tiếp theo vào hàng cao nhất của số 2, ... Đây là tổng quát hóa của "d0d2 + d1d3".

In [17]:
def minimum_sum_general(num: int) -> int:
    cnt = [0] * 10
    for c in str(num):
        cnt[int(c)] += 1
    a = b_ = 0
    turn = 0
    for digit in range(10):
        for _ in range(cnt[digit]):
            if turn == 0:
                a = a * 10 + digit
            else:
                b_ = b_ * 10 + digit
            turn ^= 1                  # đổi lượt 0 <-> 1
    return a + b_

print(minimum_sum_general(2932), minimum_sum_general(4009), minimum_sum_general(123456))

52 13 381


`turn ^= 1` dùng phép XOR để đổi qua lại giữa `0` và `1`. `a = a * 10 + digit` là công thức "**ghép thêm chữ số vào cuối**" (ngược với `% 10`).

In [18]:
a = 0
for digit in [2, 3, 9]:
    a = a * 10 + digit
    print("ghép", digit, "->", a)
t = 0
for _ in range(5):
    t ^= 1
    print("turn =", t)

ghép 2 -> 2
ghép 3 -> 23
ghép 9 -> 239
turn = 1
turn = 0
turn = 1
turn = 0
turn = 1


## 10. Vì sao đúng? (chứng minh)

Gọi bốn chữ số đã sắp `d0 ≤ d1 ≤ d2 ≤ d3`.

**Bước 1 — chỉ cần xét hai số 2 chữ số.** Các dạng chia: (1+3), (2+2), (3+1) chữ số.
- Dạng (1+3): `x + "abc"` = `100a + 10b + c + x`. Đổi thành `"bc" + "ax"` = `10b + c + 10a + x`. Hiệu = `90a ≥ 0`. Vậy dạng (2+2) luôn không tệ hơn.

**Bước 2 — trong dạng (2+2).** Tổng = `10·(t1 + t2) + (u1 + u2)` với `t` là hai chữ số hàng chục, `u` là hai chữ số hàng đơn vị. Vì `t1 + t2 + u1 + u2` là hằng số `S`, tổng = `9·(t1 + t2) + S`, nhỏ nhất khi `t1 + t2` nhỏ nhất ⇒ chọn `t1, t2 = d0, d1`.

⇒ Đáp án = `10·(d0 + d1) + d2 + d3`. ∎

Kiểm chứng bằng brute force trên **mọi** số 4 chữ số ở phần test.

## 11. Độ phức tạp

Số chữ số cố định là 4 → mọi cách đều `O(1)`. Để so sánh có ý nghĩa, xét tổng quát `d` chữ số:

| Cách | Time (theo `d`) | Space | Với bài này |
|---|---|---|---|
| 1. Brute force | `O(d! · d²)` | `O(d)` | 72 ứng viên — vẫn nhanh |
| 2. Sort chữ số (chuỗi) | `O(d log d)` | `O(d)` | `O(1)` |
| 3. `divmod` + công thức | `O(d log d)` | `O(d)` | `O(1)` |
| 4. Counting | `O(d + 10)` | `O(10)` | `O(1)` |

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Vì chỉ có 9000 input hợp lệ (`1000..9999`), ta có thể **kiểm tra toàn bộ** miền input bằng brute force — test mạnh nhất có thể!

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<20} PASS ({actual})")

impls = [minimum_sum_sort, minimum_sum_math, minimum_sum_general, Solution().minimumSum]
cases = [
    ("đề ví dụ 1", 2932, 52),
    ("đề ví dụ 2", 4009, 13),
    ("nhỏ nhất 1000", 1000, 1),
    ("lớn nhất 9999", 9999, 198),
    ("toàn chữ số khác nhau", 1234, 37),   # 13 + 24
    ("có hai số 0", 1010, 2),             # 01 + 01
    ("chữ số giảm dần", 9876, 147),        # 68 + 79
]
for name, num, exp in cases:
    for f in impls:
        assert f(num) == exp, (name, f)
    check(name, minimum_sum_sort(num), exp)

đề ví dụ 1           PASS (52)
đề ví dụ 2           PASS (13)
nhỏ nhất 1000        PASS (1)
lớn nhất 9999        PASS (198)
toàn chữ số khác nhau PASS (37)
có hai số 0          PASS (2)
chữ số giảm dần      PASS (147)


In [20]:
mismatch = 0
for num in range(1000, 10000):
    expected = minimum_sum_bruteforce(num)
    for f in impls:
        if f(num) != expected:
            mismatch += 1
print("Đã kiểm tra toàn bộ 9000 số 4 chữ số, số lần sai:", mismatch)
assert mismatch == 0

Đã kiểm tra toàn bộ 9000 số 4 chữ số, số lần sai: 0


## 13. Lỗi người mới hay gặp

### Lỗi 1: Ghép `d0d1 + d2d3` (hai chữ số nhỏ vào **cùng một số**)

In [21]:
def wrong_pairing(num):
    d = sorted(int(c) for c in str(num))
    return (d[0] * 10 + d[1]) + (d[2] * 10 + d[3])

print("sai :", wrong_pairing(2932), "| đúng:", minimum_sum_sort(2932))

sai : 61 | đúng: 52


**Vì sao sai:** `d2` lại nằm ở **hàng chục** của số thứ hai (`39`), bị nhân 10. **Sửa:** hai chữ số nhỏ nhất phải ở **hai hàng chục khác nhau**: `d0d2 + d1d3`.

### Lỗi 2: Bỏ mất chữ số 0 khi tách bằng `while x > 0` trên số đã bị "cắt"

In [22]:
def wrong_digits(num):
    digits = []
    while num > 0:
        digits.append(num % 10)
        num //= 10
    return digits

print(wrong_digits(4009), "<- đủ 4 chữ số, OK")
print(wrong_digits(int("0409")), "<- nếu input là '0409' dạng số thì mất số 0 đầu!")

[9, 0, 0, 4] <- đủ 4 chữ số, OK
[9, 0, 4] <- nếu input là '0409' dạng số thì mất số 0 đầu!


**Vì sao lưu ý:** với constraint `num >= 1000` thì `while` vẫn đủ 4 chữ số; nhưng nếu đề biến thể cho phép số 0 ở đầu (ví dụ input là chuỗi `"0409"`), `int` sẽ làm mất chữ số. **Sửa:** dùng `for _ in range(4)` hoặc làm việc trực tiếp trên chuỗi.

### Lỗi 3: So sánh chữ số dạng **chuỗi** rồi nhân với 10

In [23]:
d = sorted(str(2932))
print(d)
print("d[0] * 10 =", repr(d[0] * 10), "<- nhân chuỗi lặp 10 lần, không phải số!")

['2', '2', '3', '9']
d[0] * 10 = '2222222222' <- nhân chuỗi lặp 10 lần, không phải số!


**Vì sao sai:** `'2' * 10` là chuỗi `'2222222222'`. **Sửa:** đổi sang `int` trước: `sorted(int(c) for c in str(num))`. (Sắp chuỗi một chữ số `'0'..'9'` thì thứ tự vẫn đúng, nhưng phép toán thì không.)

### Lỗi 4: Nghĩ rằng ghép `d0d1d2 + d3` hay `d0 + d1d2d3` có thể tốt hơn

In [24]:
d = [2, 2, 3, 9]
print("d0 + d1d2d3 =", 2 + 239, "| d0d1d2 + d3 =", 223 + 9, "| đúng (2+2):", 23 + 29)

d0 + d1d2d3 = 241 | d0d1d2 + d3 = 232 | đúng (2+2): 52


Đã chứng minh ở mục "Vì sao đúng": dạng (1+3) luôn kém hơn hoặc bằng (2+2).

## 14. Mẹo nhận dạng pattern

- "Sắp xếp chữ số để tổng/hiệu **nhỏ nhất/lớn nhất**" → nghĩ về **giá trị vị trí**: chữ số ở vị trí cao quan trọng hơn → greedy đặt chữ số nhỏ (hoặc lớn) vào vị trí cao.
- "Chia đều cho hai số" → **xen kẽ** (alternate) chữ số đã sắp.
- Miền input nhỏ (9000 số) → **kiểm tra toàn bộ** bằng brute force.
- Hai cách tách chữ số: chuỗi (`str`/`int`) dễ đọc; toán (`% 10`, `// 10`, `divmod`) không tốn chuỗi.

Bài liên quan: #2578 Split With Minimum Sum (bản tổng quát), #179 Largest Number, #670 Maximum Swap, #1281 Subtract the Product and Sum of Digits of an Integer.

## 15. Biến thể

### 15.1. LeetCode 2578 — Split With Minimum Sum (số chữ số bất kỳ)
Chính là Cách 4 (xen kẽ chữ số đã sắp). Viết gọn bằng slicing bước 2:

In [25]:
def split_num(num: int) -> int:
    s = "".join(sorted(str(num)))
    return int(s[::2]) + int(s[1::2])

print(split_num(4325), split_num(687), split_num(2932))

59 75 52


`s[::2]` lấy các ký tự chỉ số chẵn (0, 2, 4, ...), `s[1::2]` lấy chỉ số lẻ — chính là "xen kẽ".

### 15.2. Tổng **lớn nhất** khi chia thành hai số
Đặt chữ số lớn vào vị trí cao: tạo một số 3 chữ số với 3 chữ số lớn nhất (giảm dần) và số còn lại là chữ số nhỏ nhất.

In [26]:
def maximum_sum_bruteforce(num):
    best = 0
    for perm in permutations(str(num)):
        t = "".join(perm)
        for p in range(1, 4):
            best = max(best, int(t[:p]) + int(t[p:]))
    return best

def maximum_sum(num):
    d = sorted((int(c) for c in str(num)), reverse=True)
    return d[0] * 100 + d[1] * 10 + d[2] + d[3]

for num in (2932, 4009, 1234):
    print(num, maximum_sum(num), maximum_sum_bruteforce(num))
assert all(maximum_sum(n) == maximum_sum_bruteforce(n) for n in range(1000, 10000, 7))
print("maximum_sum khớp brute force")

2932 934 934
4009 940 940
1234 433 433
maximum_sum khớp brute force


## 16. Bài tập (TODO — tự làm, đáp án ở cuối)

### Bài 1 — Tính tay
`num = 5381`: sắp chữ số, ghép `new1`, `new2`, tính tổng.

### Bài 2 — Đoán output
1. `divmod(4009, 10)` = ?
2. `"".join(sorted("2932"))[1::2]` = ?
3. `10 * (0 + 0) + 4 + 9` = ?

### Bài 3 — Điền chỗ trống (Cách 2)

In [27]:
def exercise_minimum_sum(num: int) -> int:
    # TODO 1: tách 4 chữ số thành list int
    # TODO 2: sắp tăng dần
    # TODO 3: trả 10 * (d0 + d1) + d2 + d3
    pass

# assert exercise_minimum_sum(2932) == 52
# assert exercise_minimum_sum(4009) == 13

### Bài 4 — Trả về **cặp** `(new1, new2)` dạng chuỗi (giữ số 0 ở đầu)
Ví dụ `4009` → `("04", "09")`.

In [28]:
def exercise_pair_strings(num: int) -> tuple[str, str]:
    # TODO
    pass

# assert exercise_pair_strings(4009) == ("04", "09")
# assert exercise_pair_strings(2932) == ("23", "29")

### Bài 5 — Biến thể: chia thành **ba** số
Cho số có đúng 6 chữ số, chia thành 3 số (dùng hết chữ số) để tổng nhỏ nhất.

In [29]:
def exercise_min_sum_three(num: int) -> int:
    # TODO: tổng quát hóa xen kẽ: chữ số thứ i vào số thứ (i % 3)
    pass

# assert exercise_min_sum_three(123456) == 14 + 25 + 36

### Bài 6 — Mini challenge: hiệu nhỏ nhất
Chia 4 chữ số thành **hai số hai chữ số** sao cho `|new1 - new2|` nhỏ nhất. (Gợi ý: brute force là đủ, nhưng hãy thử tự suy luận trước.)

In [30]:
def exercise_min_difference(num: int) -> int:
    # TODO
    pass

# assert exercise_min_difference(2932) == 3     # 32 - 29
# assert exercise_min_difference(1000) == 1     # 01 - 00

### Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Sắp `1,3,5,8` | hàng chục: 1 và 3 | `15 + 38` |
| 2 | `divmod` trả (thương, dư) | `sorted("2932")` = `['2','2','3','9']` | lấy chỉ số 1, 3 |
| 3 | Dùng `str` hoặc `divmod` | `sorted(...)` | unpack `d0, d1, d2, d3` |
| 4 | Làm trên chuỗi đã sắp | `s = "".join(sorted(str(num)))` | `(s[0] + s[2], s[1] + s[3])` |
| 5 | Ba chữ số nhỏ nhất lên hàng chục | `s[0::3], s[1::3], s[2::3]` | cộng `int(...)` của ba phần |
| 6 | Thử mọi hoán vị, cắt tại 2 | `abs(int(t[:2]) - int(t[2:]))` | lấy `min` |

## 17. Đáp án tham khảo (tự làm trước rồi hãy xem)

**Bài 1:** `5381` → `1,3,5,8` → `15 + 38 = 53`.

**Bài 2:** `(400, 9)`; `"29"`; `13`.

**Bài 6:** với `2932`, cặp `32` và `29` cho hiệu `3` là nhỏ nhất. Với `1000`: các cặp có thể là `10/00` hoặc `01/00` → hiệu nhỏ nhất `1` (`01 - 00`). Chú ý số 0 ở đầu!

In [31]:
print(minimum_sum_sort(5381), divmod(4009, 10), "".join(sorted("2932"))[1::2], 10 * (0 + 0) + 4 + 9)

53 (400, 9) 29 13


In [32]:
def answer_minimum_sum(num):
    d0, d1, d2, d3 = sorted(int(c) for c in str(num))
    return 10 * (d0 + d1) + d2 + d3


def answer_pair_strings(num):
    s = "".join(sorted(str(num)))
    return (s[0] + s[2], s[1] + s[3])


def answer_min_sum_three(num):
    s = "".join(sorted(str(num)))
    return sum(int(s[i::3]) for i in range(3))


def answer_min_difference(num):
    best = None
    for perm in permutations(str(num)):
        t = "".join(perm)
        diff = abs(int(t[:2]) - int(t[2:]))
        if best is None or diff < best:
            best = diff
    return best


assert answer_minimum_sum(2932) == 52
assert answer_minimum_sum(4009) == 13
assert answer_pair_strings(4009) == ("04", "09")
assert answer_pair_strings(2932) == ("23", "29")
assert answer_min_sum_three(123456) == 14 + 25 + 36
assert answer_min_difference(2932) == 3
assert answer_min_difference(1000) == 1
for num in range(1000, 10000, 37):
    assert answer_minimum_sum(num) == minimum_sum_bruteforce(num)
print("Đáp án tham khảo: PASS toàn bộ test!")

Đáp án tham khảo: PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def minimumSum(self, num: int) -> int:
        d0, d1, d2, d3 = sorted(int(c) for c in str(num))
        return 10 * (d0 + d1) + d2 + d3
```

**Khung tư duy "sắp chữ số":** tách chữ số → sắp → vị trí cao nhận chữ số nhỏ (bài min) / lớn (bài max) → xen kẽ khi chia nhiều số → kiểm chứng bằng brute force.

**Checklist trước khi nộp:**
- [ ] Đổi chữ số sang `int` trước khi nhân?
- [ ] Hai chữ số nhỏ nhất ở **hai hàng chục khác nhau**?
- [ ] Số 0 ở đầu được tính đúng (`04 = 4`)?
- [ ] Đã kiểm tra `1000`, `9999`, `4009`?